# Descarga dinamica FOV 8 y mascara de galaxia central

Este notebook descarga una muestra balanceada de galaxias MaNGA con un campo de vision de $8 \times PETRO_TH90$ y calcula una mascara de la galaxia central. El circulo usado para estimar el color de galaxia tiene radio fijo de 20 px.

La descarga es reanudable: con `OVERWRITE = False`, los cutouts ya presentes se reutilizan.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'src' / 'config.py').is_file() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'config.py').is_file():
    raise FileNotFoundError('No se encontro la raiz del proyecto (src/config.py).')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CUTOUT_SIZE_PX, RANDOM_SEED
from src.data.cutout_downloader import build_fov_jobs, download_batch
from src.data.galaxy_mask import apply_mask, galaxy_color_mask
from src.data.morfologia import load_labeled_sample

## Configuracion

`CENTER_RADIUS_PX` se convierte a la fraccion requerida por `galaxy_color_mask`, por lo que el radio de muestreo se mantiene en 20 px para cutouts de 256 px. La configuracion de descarga replica el mecanismo del explorador FOV: jobs explicitos, concurrencia limitada y ritmo global de peticiones.

In [ ]:
SAMPLE_SIZE = 20
FOV_FACTOR = 8.0
CENTER_RADIUS_PX = 20
DOWNLOAD_TAG = 'dynamic_fov8'
LAYER = 'ls-dr11'
BANDS = 'grz'
MAX_CONCURRENT_REQUESTS = 2
MIN_REQUEST_INTERVAL_S = 1.0
TIMEOUT_S = 60
MAX_RETRIES = 3
OVERWRITE = False

if SAMPLE_SIZE < 2 or SAMPLE_SIZE % 2:
    raise ValueError('SAMPLE_SIZE debe ser un entero par mayor o igual a 2.')
if FOV_FACTOR <= 0:
    raise ValueError('FOV_FACTOR debe ser positivo.')
if not 0 < CENTER_RADIUS_PX < CUTOUT_SIZE_PX / 2:
    raise ValueError('CENTER_RADIUS_PX debe caber dentro del cutout.')

CENTER_FRAC = CENTER_RADIUS_PX / CUTOUT_SIZE_PX
DOWNLOAD_DIR = PROJECT_ROOT / 'data' / 'legacy_cutouts_playground' / DOWNLOAD_TAG / 'fov8'
MASKED_DIR = PROJECT_ROOT / 'data' / 'legacy_cutouts_playground' / DOWNLOAD_TAG / 'fov8_masked'
INDEX_CSV = PROJECT_ROOT / 'data' / 'legacy_cutouts_playground' / DOWNLOAD_TAG / 'download_index.csv'

print(f'Muestra: {SAMPLE_SIZE} galaxias balanceadas')
print(f'FOV: {FOV_FACTOR} x PETRO_TH90')
print(f'Radio central: {CENTER_RADIUS_PX} px ({CENTER_FRAC:.3f} del lado)')
print(f'Salida: {DOWNLOAD_DIR}')

## Seleccion y descarga

La muestra es balanceada entre galaxias tempranas y tardias, reproducible mediante `RANDOM_SEED` y limitada a objetos con `PETRO_TH90` valido. Se construye un job por galaxia y `download_batch` los descarga con concurrencia acotada, rate limiting compartido, reintentos y validacion del JPEG.

In [ ]:
sample = load_labeled_sample(n=SAMPLE_SIZE, seed=RANDOM_SEED)
display(sample[['name', 'label', 'Ttype', 'PETRO_TH90']])
print(sample['label'].value_counts().to_string())

jobs = build_fov_jobs(
    sample,
    output_dir=DOWNLOAD_DIR,
    fov_factor=FOV_FACTOR,
    layer=LAYER,
    bands=BANDS,
)
download_index = download_batch(
    jobs,
    max_concurrent=MAX_CONCURRENT_REQUESTS,
    min_request_interval_s=MIN_REQUEST_INTERVAL_S,
    overwrite=OVERWRITE,
    timeout_s=TIMEOUT_S,
    max_retries=MAX_RETRIES,
)
download_index['file_exists'] = download_index['local_path'].map(lambda path: Path(path).is_file())
INDEX_CSV.parent.mkdir(parents=True, exist_ok=True)
download_index.to_csv(INDEX_CSV, index=False)
display(download_index[['name', 'label', 'PETRO_TH90', 'pixscale', 'fov_effective_arcsec', 'status', 'file_exists', 'error']])
print(f'Indice guardado en {INDEX_CSV}')

## Calculo y guardado de mascaras

Para cada imagen descargada, se estima el color de la galaxia con el circulo central de 20 px, se distingue del cielo del borde y se conserva la componente conexa central. Los resultados se guardan con el fondo en negro.

In [ ]:
MASKED_DIR.mkdir(parents=True, exist_ok=True)
mask_records = []

for record in download_index.itertuples(index=False):
    source_path = Path(record.local_path)
    output_path = MASKED_DIR / f'{record.name}.jpg'
    if record.status == 'error' or not source_path.is_file():
        mask_records.append({'name': record.name, 'mask_coverage': np.nan, 'masked_path': None, 'mask_status': 'skipped'})
        continue

    with Image.open(source_path) as source_image:
        image = np.asarray(source_image.convert('RGB'))
    mask = galaxy_color_mask(image, center_frac=CENTER_FRAC)
    Image.fromarray(apply_mask(image, mask)).save(output_path, quality=95)
    mask_records.append({
        'name': record.name,
        'mask_coverage': float(mask.mean()),
        'masked_path': str(output_path),
        'mask_status': 'saved',
    })

mask_index = pd.DataFrame(mask_records)
results = download_index.merge(mask_index, on='name', how='left')
results.to_csv(INDEX_CSV, index=False)
display(results[['name', 'label', 'status', 'mask_status', 'mask_coverage', 'masked_path']])

## Revision visual

Cada fila muestra la imagen original, la mascara binaria y la galaxia aislada. Las coberturas extremas deben revisarse visualmente.

In [ ]:
valid_results = results[results['mask_status'].eq('saved')].reset_index(drop=True)
if valid_results.empty:
    raise RuntimeError('No hay mascaras disponibles para visualizar.')

fig, axes = plt.subplots(len(valid_results), 3, figsize=(10, 3.2 * len(valid_results)), squeeze=False)
for axis_row, record in zip(axes, valid_results.itertuples(index=False)):
    with Image.open(Path(record.local_path)) as source_image:
        image = np.asarray(source_image.convert('RGB'))
    mask = galaxy_color_mask(image, center_frac=CENTER_FRAC)

    axis_row[0].imshow(image)
    axis_row[0].set_title(f'{record.name}: original')
    axis_row[1].imshow(mask, cmap='gray', vmin=0, vmax=1)
    axis_row[1].set_title(f'mascara ({record.mask_coverage:.1%})')
    axis_row[2].imshow(apply_mask(image, mask))
    axis_row[2].set_title('galaxia aislada')
    for axis in axis_row:
        axis.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
display(results['mask_coverage'].describe().to_frame('mask_coverage'))
print('Menor cobertura:')
display(results.nsmallest(5, 'mask_coverage')[['name', 'label', 'mask_coverage']])
print('Mayor cobertura:')
display(results.nlargest(5, 'mask_coverage')[['name', 'label', 'mask_coverage']])

## Inspeccion visual de FOV, mascara y radio objetivo

Cada imagen FOV 8 se recorta centrada y se reescala para simular los FOV `[8, 6, 5, 4, 3, 2, 1]`. La mascara se recalcula en cada simulacion. El circulo de radio 70 px es verde cuando la mascara cubre al menos 25% de su anillo angular y rojo cuando no lo cubre.

In [ ]:
ESTIMATION_FOVS = [8, 7, 6, 5, 4, 3, 2, 1]
TARGET_RADIUS_PX = 75
RING_HALF_WIDTH_PX = 4
N_SECTORS = 72
MIN_ANGULAR_COVERAGE = 0.25
CENTER_RADIUS_PX = 20

CENTER_FRAC = CENTER_RADIUS_PX / CUTOUT_SIZE_PX


def simulate_fov_from_base(image, base_fov_factor, target_fov_factor, output_size_px):
    if target_fov_factor > base_fov_factor:
        raise ValueError('El FOV objetivo no puede ser mayor que el FOV base.')

    source = image.convert('RGB')
    width, height = source.size
    crop_fraction = target_fov_factor / base_fov_factor
    crop_side_px = max(1, int(round(min(width, height) * crop_fraction)))
    left = (width - crop_side_px) // 2
    top = (height - crop_side_px) // 2
    crop = source.crop((left, top, left + crop_side_px, top + crop_side_px))
    return crop.resize((output_size_px, output_size_px), Image.Resampling.LANCZOS), crop_side_px

def simulate_mask_from_base(mask, base_fov_factor, target_fov_factor, output_size_px):
    height, width = mask.shape
    crop_fraction = target_fov_factor / base_fov_factor
    crop_side_px = max(1, int(round(min(width, height) * crop_fraction)))
    left = (width - crop_side_px) // 2
    top = (height - crop_side_px) // 2

    mask_image = Image.fromarray(mask.astype(np.uint8) * 255)
    crop = mask_image.crop((left, top, left + crop_side_px, top + crop_side_px))
    resized = crop.resize(
        (output_size_px, output_size_px),
        Image.Resampling.NEAREST,
    )
    return np.asarray(resized, dtype=bool)


def angular_ring_coverage(mask, target_radius_px, ring_half_width_px, n_sectors):
    height, width = mask.shape
    yy, xx = np.mgrid[:height, :width]
    center_y = (height - 1) / 2
    center_x = (width - 1) / 2
    radius = np.hypot(yy - center_y, xx - center_x)
    ring = np.abs(radius - target_radius_px) <= ring_half_width_px
    angles = (np.arctan2(yy - center_y, xx - center_x) + 2 * np.pi) % (2 * np.pi)
    sectors = np.floor(angles * n_sectors / (2 * np.pi)).astype(int)
    occupied_sectors = np.unique(sectors[ring & mask])
    return len(occupied_sectors) / n_sectors


if valid_results.empty:
    raise RuntimeError('No hay cutouts validos para inspeccionar.')

inspection_records = []
for record in valid_results.itertuples(index=False):
    with Image.open(Path(record.local_path)) as source_image:
        base_image = source_image.convert('RGB')

    base_array = np.asarray(base_image)
    base_mask = galaxy_color_mask(base_array, center_frac=CENTER_FRAC)

    fig, axes = plt.subplots(1, len(ESTIMATION_FOVS), figsize=(3.2 * len(ESTIMATION_FOVS), 3.8), squeeze=False)
    for axis, factor in zip(axes[0], ESTIMATION_FOVS):
        simulated, crop_side_px = simulate_fov_from_base(base_image, FOV_FACTOR, factor, CUTOUT_SIZE_PX)
        simulated_array = np.asarray(simulated)
        mask = simulate_mask_from_base(
                base_mask,
                FOV_FACTOR,
                factor,
                CUTOUT_SIZE_PX,
            )
        coverage = angular_ring_coverage(mask, TARGET_RADIUS_PX, RING_HALF_WIDTH_PX, N_SECTORS)
        reaches_target = coverage >= MIN_ANGULAR_COVERAGE
        circle_color = 'limegreen' if reaches_target else 'red'

        axis.imshow(simulated_array)
        #axis.imshow(np.ma.masked_where(~mask, mask), cmap='cool', alpha=0.35, vmin=0, vmax=1)
        #axis.add_patch(plt.Circle(((CUTOUT_SIZE_PX - 1) / 2, (CUTOUT_SIZE_PX - 1) / 2), TARGET_RADIUS_PX, color=circle_color, fill=False, linewidth=1.6))
        axis.set_title(f'FOV {factor:g}\ncobertura {coverage:.0%}', color=circle_color, fontsize=9)
        axis.set_xticks([])
        axis.set_yticks([])

        inspection_records.append({
            'name': record.name,
            'label': record.label,
            'fov_factor': factor,
            'crop_side_px': crop_side_px,
            'angular_coverage': coverage,
            'reaches_target': reaches_target,
        })

    fig.suptitle(f'{record.name} ({record.label}): mascara cian y circulo objetivo', y=1.03)
    plt.tight_layout()
    plt.show()

inspection_index = pd.DataFrame(inspection_records)
recommended_fov = (
    inspection_index[inspection_index['reaches_target']]
    .sort_values(['name', 'fov_factor'], ascending=[True, False])
    .groupby('name', as_index=False)
    .first()[['name', 'label', 'fov_factor', 'angular_coverage']]
    .rename(columns={'fov_factor': 'fov_recomendado', 'angular_coverage': 'cobertura_angular'})
)
display(inspection_index)
display(recommended_fov)

In [ ]:
from src.config import MORFOLOGIA_CSV
from src.data.morfologia import derive_ttype_label, load_morfologia

TEST_GALAXIES = [
    "manga-8561-3704",
    "manga-8147-3704",
    "manga-8938-3701",
    "manga-8978-12705",
    "manga-8990-9101",
    "manga-9028-12705",
    "manga-9882-12705",
    "manga-10515-12704",
    "manga-10837-12705",
    "manga-10839-3704",
    "manga-11020-9102",
    "manga-11759-6102",
    "manga-11960-1901",
    "manga-11965-12705",
    "manga-12078-12703",
    "manga-12769-1902",
    "manga-7962-6102",
    "manga-10001-12703"
]

ESTIMATION_FOVS = [8, 7, 6, 5, 4, 3, 2, 1]
TARGET_RADIUS_PX = 75
RING_HALF_WIDTH_PX = 4
N_SECTORS = 72
MIN_ANGULAR_COVERAGE = 0.25

CENTER_RADIUS_PX = 20

CENTER_FRAC = CENTER_RADIUS_PX / CUTOUT_SIZE_PX

show_circle = False
show_mask = False

catalog = derive_ttype_label(load_morfologia(MORFOLOGIA_CSV))
catalog = catalog.drop_duplicates("name").set_index("name")

missing_names = [name for name in TEST_GALAXIES if name not in catalog.index]
test_sample = catalog.reindex(
    [name for name in TEST_GALAXIES if name in catalog.index]
).reset_index()

if missing_names:
    print(f"No encontrados en el catalogo: {missing_names}")
if test_sample.empty:
    raise ValueError("Ninguna galaxia de TEST_GALAXIES existe en el catalogo.")

test_jobs = build_fov_jobs(
    test_sample,
    output_dir=DOWNLOAD_DIR,
    fov_factor=FOV_FACTOR,
    layer=LAYER,
    bands=BANDS,
)

test_download_index = download_batch(
    test_jobs,
    max_concurrent=MAX_CONCURRENT_REQUESTS,
    min_request_interval_s=MIN_REQUEST_INTERVAL_S,
    overwrite=OVERWRITE,
    timeout_s=TIMEOUT_S,
    max_retries=MAX_RETRIES,
)

test_download_index["file_exists"] = test_download_index["local_path"].map(
    lambda path: Path(path).is_file()
)
test_valid = test_download_index[
    test_download_index["status"].ne("error")
    & test_download_index["file_exists"]
].reset_index(drop=True)

if test_valid.empty:
    raise RuntimeError("No hubo descargas validas para inspeccionar.")

test_records = []

for record in test_valid.itertuples(index=False):
    with Image.open(Path(record.local_path)) as source_image:
        base_image = source_image.convert("RGB")

    base_mask = galaxy_color_mask(
        np.asarray(base_image),
        center_frac=CENTER_FRAC,
    )

    fig, axes = plt.subplots(
        1,
        len(ESTIMATION_FOVS),
        figsize=(3.2 * len(ESTIMATION_FOVS), 3.8),
        squeeze=False,
    )

    for axis, factor in zip(axes[0], ESTIMATION_FOVS):
        simulated, crop_side_px = simulate_fov_from_base(
            base_image,
            FOV_FACTOR,
            factor,
            CUTOUT_SIZE_PX,
        )
        mask = simulate_mask_from_base(
            base_mask,
            FOV_FACTOR,
            factor,
            CUTOUT_SIZE_PX,
        )

        coverage = angular_ring_coverage(
            mask,
            TARGET_RADIUS_PX,
            RING_HALF_WIDTH_PX,
            N_SECTORS,
        )
        reaches_target = coverage >= MIN_ANGULAR_COVERAGE
        circle_color = "limegreen" if reaches_target else "red"

        axis.imshow(np.asarray(simulated))
        if show_mask:
            axis.imshow(
                np.ma.masked_where(~mask, mask),
                cmap="cool",
                alpha=0.35,
                vmin=0,
                vmax=1,
            )
        if show_circle:
            axis.add_patch(
                plt.Circle(
                    ((CUTOUT_SIZE_PX - 1) / 2, (CUTOUT_SIZE_PX - 1) / 2),
                    TARGET_RADIUS_PX,
                    color=circle_color,
                    fill=False,
                    linewidth=1.6,
                )
            )
        axis.set_title(
            f"FOV {factor:g}\ncobertura {coverage:.0%}",
            color=circle_color,
            fontsize=9,
        )
        axis.set_xticks([])
        axis.set_yticks([])

        test_records.append(
            {
                "name": record.name,
                "label": record.label,
                "fov_factor": factor,
                "crop_side_px": crop_side_px,
                "angular_coverage": coverage,
                "reaches_target": reaches_target,
            }
        )

    fig.suptitle(
        f"{record.name} ({record.label}): prueba de caso extremo",
        y=1.03,
    )
    plt.tight_layout()
    plt.show()

test_inspection_index = pd.DataFrame(test_records)
display(test_inspection_index)

## Dataset consolidado: descarga y calidad visual

Esta sección reutiliza la muestra descargada en el notebook. Conserva los parámetros, estado y rutas de cada descarga, y agrega las métricas y banderas de `analyze_image_quality` para cada JPEG disponible. No genera revisión visual adicional.

In [ ]:
from src.data import analyze_image_quality

QUALITY_INDEX_CSV = INDEX_CSV.with_name('download_quality_index.csv')
quality_records = []

for record in results.itertuples(index=False):
    source_path = Path(record.local_path)
    base_record = {
        'name': record.name,
        'quality_status': 'skipped',
        'quality_error': None,
    }

    if record.status == 'error' or not source_path.is_file():
        base_record['quality_error'] = 'JPEG no disponible para análisis.'
        quality_records.append(base_record)
        continue

    try:
        with Image.open(source_path) as source_image:
            rgb_normalized = np.asarray(source_image.convert('RGB'), dtype=np.float32) / 255.0
        report = analyze_image_quality(rgb_normalized)
    except Exception as error:
        base_record['quality_status'] = 'error'
        base_record['quality_error'] = str(error)
        quality_records.append(base_record)
        continue

    quality_records.append({
        **base_record,
        'quality_status': 'analyzed',
        'quality_line_flag': report['line_artifact'],
        'quality_line_score': report['line_score'],
        'quality_bright_flag': report['bright_contamination'],
        'quality_bright_score': report['bright_score'],
        'quality_saturation_flag': report['saturation'],
        'quality_saturation_score': report['saturation_score'],
        'quality_is_suspicious': report['is_suspicious'],
        'quality_reasons': ' | '.join(report['reasons']),
        **report['metrics'],
    })

quality_index = pd.DataFrame(quality_records)
download_quality_dataset = results.merge(quality_index, on='name', how='left')
download_quality_dataset.to_csv(QUALITY_INDEX_CSV, index=False)

download_columns = [
    column for column in [
        'name', 'label', 'PETRO_TH90', 'layer', 'bands_or_color_mode',
        'fov_factor', 'fov_requested_arcsec', 'fov_effective_arcsec',
        'pixscale', 'pixscale_clamped', 'size_px', 'status', 'file_exists',
    ] if column in download_quality_dataset.columns
]
quality_columns = [
    column for column in [
        'quality_status', 'quality_is_suspicious', 'quality_line_flag',
        'quality_line_score', 'quality_bright_flag', 'quality_bright_score',
        'quality_saturation_flag', 'quality_saturation_score',
        'saturated_fraction', 'largest_outer_component_fraction',
        'quality_reasons', 'quality_error',
    ] if column in download_quality_dataset.columns
]

display(download_quality_dataset[download_columns + quality_columns])
print(f'Dataset consolidado guardado en {QUALITY_INDEX_CSV}')

In [ ]:
download_quality_dataset[download_quality_dataset["quality_is_suspicious"] == True]

# Sección para modificar fov factor estimado por fov_factor_estimator.py

In [ ]:
import pandas as pd

sample = pd.read_csv("../../data/morfologia_curada.csv")
fov_estimations = pd.read_csv("../../data/datasets/morfologia_curada/download_fov_estimations.csv")

sample["PETRO_TH90"] = pd.to_numeric(
    sample["PETRO_TH90"],
    errors="coerce",
)

invalid_petro = ~sample["PETRO_TH90"].gt(0)
#print(f"Filas excluidas por PETRO_TH90 inválido: {invalid_petro.sum()}")

sample = sample.loc[~invalid_petro].copy()

sample = pd.merge(
    sample,
    fov_estimations[["name","fov_factor","reaches_target"]],
    on="name",
    how="inner"
)

sample["fov_factor"] = sample["fov_factor"].astype(float)

#sample.loc[sample["fov_factor"] == 1.0, "fov_factor"] = 0.1
#sample.loc[sample["fov_factor"] == 2.0, "fov_factor"] = 0.3
#sample.loc[sample["fov_factor"] == 3.0, "fov_factor"] = 0.5
sample["fov_factor"] = sample["fov_factor"].apply(lambda x: x - 3 if x > 3 else x)
#sample["fov_factor"] = sample["fov_factor"] + 3.0

sample["fov_factor"] = sample["fov_factor"].astype(float)

In [57]:
sample["fov_factor"].value_counts()

fov_factor
2.0    3267
1.0    2814
3.0    1709
4.0     880
0.5     674
5.0     577
0.3      78
0.1       7
Name: count, dtype: int64

In [58]:
sample.to_csv("../../data/datasets/morfologia_curada_fov_-3/morfologia_curada_with_fov.csv", index=False)

In [ ]:
sample[sample["fov_factor"]==1]

In [ ]:
# plot the distribution of fov_factor
import matplotlib.pyplot as plt

sample["fov_factor"].value_counts().plot(kind="bar")
plt.show()

In [ ]:
sample[sample["fov_factor"].isnull()]

In [ ]:
sample.isnull().sum()